# 03 — Preparación de datos y feature engineering

**Objetivo:** Transformar `ventas_semanales` en un dataset modelable con todas las features predictivas necesarias para entrenar los modelos de forecasting (LightGBM, XGBoost, Prophet). Aplicar partición temporal estricta y guardar el resultado.

**Fase CRISP-DM:** 3 — Preparación de los datos.

**Inputs:**
- Tabla `ventas_semanales` (MySQL) — 184.200 filas, granularidad SKU × centro × semana
- Tabla `dim_producto` (MySQL) — 4.282 SKUs con metadatos

**Outputs:**
- Archivo `data/processed/dataset_modelable.parquet` — dataset listo para modelado
- Tabla MySQL `dataset_features` — versión navegable desde Adminer/Tableau
- Documentación de las features creadas y de la partición temporal

**Decisiones técnicas clave:**

1. **Expansión de la serie con ceros**: cada SKU activo tiene todas las semanas, rellenando con 0 las semanas sin venta.
2. **SKUs activos únicamente**: se descartan SKUs sin venta en los últimos 90 días del periodo (para evitar predecir productos descontinuados).
3. **Sin data leakage**: las estadísticas por SKU y la clasificación ABC se calculan SOLO sobre el set de entrenamiento.
4. **Partición temporal estricta**: train (91 semanas) / valid (13 semanas) / test 2026 (13 semanas).

**Autor:** Equipo del proyecto — Diego Andrés De Jesús Montenegro y Luis David Andrade Díaz
**Fecha:** mayo 2026

In [1]:
# =========================
# Imports
# =========================
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import holidays
from sqlalchemy import text

from src.db import get_engine

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Imports completados.")

Imports completados.


In [2]:
# =========================
# Constantes del proyecto
# =========================
# Periodo de analisis (validado con auditoria del dataset)
FECHA_MIN = pd.Timestamp("2024-01-01")
FECHA_MAX = pd.Timestamp("2026-03-31")

# Particion temporal (decision documentada en docs/05_decisiones_tecnicas.md)
FECHA_FIN_TRAIN = pd.Timestamp("2025-09-30")
FECHA_FIN_VALID = pd.Timestamp("2025-12-31")
FECHA_FIN_TEST = pd.Timestamp("2026-03-31")

# SKUs activos: deben haber vendido en los ultimos N dias del periodo
DIAS_PARA_ACTIVO = 90

# Carpeta de salida
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Periodo:      {FECHA_MIN.date()} -> {FECHA_MAX.date()}")
print(f"Train:        {FECHA_MIN.date()} -> {FECHA_FIN_TRAIN.date()}")
print(f"Validation:   {(FECHA_FIN_TRAIN + pd.Timedelta(days=1)).date()} -> {FECHA_FIN_VALID.date()}")
print(f"Test 2026:    {(FECHA_FIN_VALID + pd.Timedelta(days=1)).date()} -> {FECHA_FIN_TEST.date()}")

Periodo:      2024-01-01 -> 2026-03-31
Train:        2024-01-01 -> 2025-09-30
Validation:   2025-10-01 -> 2025-12-31
Test 2026:    2026-01-01 -> 2026-03-31


## 1. Carga de datos

In [3]:
engine = get_engine()

ventas_semanales = pd.read_sql(
    "SELECT * FROM ventas_semanales",
    engine,
    parse_dates=["fecha_inicio_semana"],
)

dim_producto = pd.read_sql(
    "SELECT * FROM dim_producto",
    engine,
    parse_dates=["fecha_primera_venta", "fecha_ultima_venta"],
)

print(f"ventas_semanales: {len(ventas_semanales):>10,} filas")
print(f"dim_producto:     {len(dim_producto):>10,} SKUs")

ventas_semanales:    184,200 filas
dim_producto:          4,282 SKUs


## 2. Identificación de SKUs activos

Solo modelamos SKUs que tuvieron al menos una venta en los últimos 90 días del periodo. Los descontinuados se excluyen para no enseñarle al modelo a predecir productos que ya no existen.

In [4]:
# Identificar SKUs activos
fecha_corte_activo = FECHA_MAX - pd.Timedelta(days=DIAS_PARA_ACTIVO)
skus_activos = (
    ventas_semanales[ventas_semanales["fecha_inicio_semana"] >= fecha_corte_activo]
    ["item"].unique()
)

print(f"Fecha de corte para considerarse activo: {fecha_corte_activo.date()}")
print(f"SKUs totales en dim_producto:           {len(dim_producto):>6,}")
print(f"SKUs activos (con venta en >= {DIAS_PARA_ACTIVO} dias): {len(skus_activos):>6,}")
print(f"SKUs descartados (descontinuados):      {len(dim_producto) - len(skus_activos):>6,}")

Fecha de corte para considerarse activo: 2025-12-31
SKUs totales en dim_producto:            4,282
SKUs activos (con venta en >= 90 dias):  2,310
SKUs descartados (descontinuados):       1,972


In [5]:
# Filtrar a SKUs activos
ventas = ventas_semanales[ventas_semanales["item"].isin(skus_activos)].copy()
print(f"\nFilas tras filtrar SKUs activos: {len(ventas):,} (antes: {len(ventas_semanales):,})")
print(f"Porcentaje retenido: {len(ventas) / len(ventas_semanales) * 100:.1f}%")


Filas tras filtrar SKUs activos: 168,084 (antes: 184,200)
Porcentaje retenido: 91.3%


## 3. Expansión de la serie con ceros

Construimos el grid completo SKU × centro × semana y hacemos `LEFT JOIN` con las ventas reales. Las combinaciones sin venta se rellenan con cero.

**Importante:** solo expandimos a partir de la fecha de primera venta de cada SKU. No tiene sentido tener "ceros" antes de que el SKU existiera en el catálogo.

In [6]:
# Generar el calendario completo de semanas (inicio en lunes)
semanas_grid = pd.date_range(
    start=FECHA_MIN, end=FECHA_MAX, freq="W-MON"
).to_frame(index=False, name="fecha_inicio_semana")

print(f"Semanas del grid completo: {len(semanas_grid)}")
print(f"Primera: {semanas_grid['fecha_inicio_semana'].min().date()}")
print(f"Ultima:  {semanas_grid['fecha_inicio_semana'].max().date()}")

Semanas del grid completo: 118
Primera: 2024-01-01
Ultima:  2026-03-30


In [7]:
# Fecha de primera venta por SKU (para no tener ceros antes de su existencia)
primera_venta_por_sku = (
    ventas.groupby("item")["fecha_inicio_semana"]
    .min()
    .reset_index()
    .rename(columns={"fecha_inicio_semana": "fecha_primera_venta_sku"})
)

# Grid teorico: cada SKU activo x cada centro donde tuvo venta x cada semana
combinaciones_sku_centro = ventas[["item", "centro_operacion"]].drop_duplicates()
print(f"Combinaciones SKU x centro con al menos 1 venta: {len(combinaciones_sku_centro):,}")

Combinaciones SKU x centro con al menos 1 venta: 5,201


In [8]:
# Cross join: cada combinacion SKU x centro contra todas las semanas
grid = combinaciones_sku_centro.merge(semanas_grid, how="cross")
print(f"Grid teorico inicial: {len(grid):>10,}")

# Filtrar para no incluir semanas anteriores a la primera venta del SKU
grid = grid.merge(primera_venta_por_sku, on="item", how="left")
grid = grid[grid["fecha_inicio_semana"] >= grid["fecha_primera_venta_sku"]].copy()
grid = grid.drop(columns=["fecha_primera_venta_sku"])
print(f"Grid tras truncar por primera venta: {len(grid):>10,}")

Grid teorico inicial:    613,718
Grid tras truncar por primera venta:    517,846


In [9]:
# Hacer LEFT JOIN con las ventas reales
ventas_expandidas = grid.merge(
    ventas[["item", "centro_operacion", "fecha_inicio_semana",
            "cantidad_total", "valor_bruto_total", "num_transacciones"]],
    on=["item", "centro_operacion", "fecha_inicio_semana"],
    how="left",
)

# Rellenar ceros donde no habia venta
for col in ["cantidad_total", "valor_bruto_total", "num_transacciones"]:
    ventas_expandidas[col] = ventas_expandidas[col].fillna(0)

# Calcular columnas de tiempo
ventas_expandidas["anio"] = ventas_expandidas["fecha_inicio_semana"].dt.isocalendar().year.astype(int)
ventas_expandidas["semana"] = ventas_expandidas["fecha_inicio_semana"].dt.isocalendar().week.astype(int)

print(f"Dataset expandido: {len(ventas_expandidas):>10,} filas")
print(f"Combinaciones con venta:  {(ventas_expandidas['cantidad_total'] > 0).sum():>10,}")
print(f"Combinaciones con cero:   {(ventas_expandidas['cantidad_total'] == 0).sum():>10,}")
print(f"\nValidacion de cantidad total (debe coincidir):")
print(f"  Antes de expandir:  {ventas['cantidad_total'].sum():>15,.2f}")
print(f"  Despues de expandir: {ventas_expandidas['cantidad_total'].sum():>15,.2f}")
print(f"  Diferencia:         {ventas['cantidad_total'].sum() - ventas_expandidas['cantidad_total'].sum():>15,.4f}")

Dataset expandido:    517,846 filas
Combinaciones con venta:     168,084
Combinaciones con cero:      349,762

Validacion de cantidad total (debe coincidir):
  Antes de expandir:    23,235,265.51
  Despues de expandir:   23,235,265.51
  Diferencia:                  0.0000


## 4. Variables de calendario

Mes, semana, trimestre y variables cíclicas (seno/coseno) que ayudan al modelo a capturar estacionalidad.

In [10]:
df = ventas_expandidas.copy()

# Variables temporales basicas
df["mes"] = df["fecha_inicio_semana"].dt.month
df["trimestre"] = df["fecha_inicio_semana"].dt.quarter
df["dia_anio"] = df["fecha_inicio_semana"].dt.dayofyear

# Variables ciclicas (capturan que diciembre esta cerca de enero, etc.)
df["sin_mes"] = np.sin(2 * np.pi * df["mes"] / 12)
df["cos_mes"] = np.cos(2 * np.pi * df["mes"] / 12)
df["sin_semana"] = np.sin(2 * np.pi * df["semana"] / 52)
df["cos_semana"] = np.cos(2 * np.pi * df["semana"] / 52)

print("Variables de calendario creadas:")
print("  mes, trimestre, dia_anio, sin_mes, cos_mes, sin_semana, cos_semana")
df[["fecha_inicio_semana", "mes", "trimestre", "sin_mes", "cos_mes"]].head()

Variables de calendario creadas:
  mes, trimestre, dia_anio, sin_mes, cos_mes, sin_semana, cos_semana


,fecha_inicio_semana,mes,trimestre,sin_mes,cos_mes
0,2024-01-01,1,1,0.5000,0.8660
1,2024-01-08,1,1,0.5000,0.8660
2,2024-01-15,1,1,0.5000,0.8660
3,2024-01-22,1,1,0.5000,0.8660
4,2024-01-29,1,1,0.5000,0.8660


## 5. Festivos de Colombia

Conteo de días festivos en cada semana. Los festivos afectan la demanda en el sector de la construcción.

In [11]:
# Construir el calendario de festivos de Colombia
anios_festivos = range(FECHA_MIN.year, FECHA_MAX.year + 1)
festivos_co = holidays.Colombia(years=anios_festivos)

# Convertir a DataFrame para procesar
festivos_df = pd.DataFrame(
    [(pd.Timestamp(fecha), nombre) for fecha, nombre in festivos_co.items()],
    columns=["fecha", "festivo"]
)
festivos_df["fecha_inicio_semana"] = (
    festivos_df["fecha"] - pd.to_timedelta(festivos_df["fecha"].dt.weekday, unit="D")
).dt.normalize()

# Conteo de festivos por semana
festivos_por_semana = (
    festivos_df.groupby("fecha_inicio_semana")
    .size()
    .reset_index(name="num_festivos")
)

# Merge con el dataset
df = df.merge(festivos_por_semana, on="fecha_inicio_semana", how="left")
df["num_festivos"] = df["num_festivos"].fillna(0).astype(int)

print(f"Total festivos en el periodo: {len(festivos_df)}")
print(f"\nDistribucion de num_festivos por semana:")
print(df["num_festivos"].value_counts().sort_index())

Total festivos en el periodo: 53

Distribucion de num_festivos por semana:
num_festivos
0    360615
1    143758
2      9734
3      3739
Name: count, dtype: int64


## 6. Lags y medias móviles

Las features más importantes en forecasting: la demanda pasada predice la futura.

**Lags:** valor de `cantidad_total` hace N semanas (1, 2, 4, 8, 13, 52).
**Medias móviles:** promedio de las últimas N semanas.

⚠️ **Importante:** los lags y rollings se calculan SOBRE TODO el periodo (no solo train), porque la generación de un lag no introduce data leakage — usar el valor de hace 4 semanas para predecir el valor actual es legítimo en cualquier set.

In [12]:
# Ordenar por SKU x centro x fecha para que los lags funcionen
df = df.sort_values(["item", "centro_operacion", "fecha_inicio_semana"]).reset_index(drop=True)

# Definir el grupo
grupo = df.groupby(["item", "centro_operacion"])

# Lags
for lag in [1, 2, 4, 8, 13, 52]:
    df[f"lag_{lag}"] = grupo["cantidad_total"].shift(lag)
print("  Lags creados: lag_1, lag_2, lag_4, lag_8, lag_13, lag_52")

# Medias moviles: usamos transform() para que el resultado tenga el mismo
# indice que el df original. shift(1) evita usar la semana actual.
df["rolling_mean_4"] = grupo["cantidad_total"].transform(
    lambda s: s.shift(1).rolling(window=4, min_periods=1).mean()
)
df["rolling_mean_13"] = grupo["cantidad_total"].transform(
    lambda s: s.shift(1).rolling(window=13, min_periods=1).mean()
)
df["rolling_std_4"] = grupo["cantidad_total"].transform(
    lambda s: s.shift(1).rolling(window=4, min_periods=1).std()
)
print("  Medias moviles: rolling_mean_4, rolling_mean_13, rolling_std_4")

# Tendencia: diferencia respecto a la semana anterior
df["diff_1"] = grupo["cantidad_total"].diff(1)
print("  Tendencia: diff_1")

  Lags creados: lag_1, lag_2, lag_4, lag_8, lag_13, lag_52
  Medias moviles: rolling_mean_4, rolling_mean_13, rolling_std_4
  Tendencia: diff_1


In [13]:
# Vistazo a los lags creados
sample_sku = df["item"].iloc[0]
sample_centro = df["centro_operacion"].iloc[0]
print(f"Ejemplo: SKU {sample_sku} en centro {sample_centro}")
df[(df["item"] == sample_sku) & (df["centro_operacion"] == sample_centro)][
    ["fecha_inicio_semana", "cantidad_total", "lag_1", "lag_4", "rolling_mean_4"]
].head(10)

Ejemplo: SKU 000001 en centro 001


,fecha_inicio_semana,cantidad_total,lag_1,lag_4,rolling_mean_4
0,2024-01-01,1.0000,NaN,NaN,NaN
1,2024-01-08,1.0000,1.0000,NaN,1.0000
2,2024-01-15,5.0000,1.0000,NaN,1.0000
3,2024-01-22,0.0000,5.0000,NaN,2.3333
4,2024-01-29,0.0000,0.0000,1.0000,1.7500
5,2024-02-05,2.0000,0.0000,1.0000,1.5000
6,2024-02-12,3.0000,2.0000,5.0000,1.7500
7,2024-02-19,2.0000,3.0000,0.0000,1.2500
8,2024-02-26,5.0000,2.0000,0.0000,1.7500
9,2024-03-04,4.0000,5.0000,2.0000,3.0000


## 7. Clasificación ABC/XYZ preliminar (solo con train)

Esta clasificación se usa como feature del modelo. **Solo se calcula con datos de train** para evitar data leakage.

La clasificación formal y exhaustiva está en el notebook 04.

In [14]:
# Conjunto de entrenamiento (solo para calcular estadisticas)
mask_train = df["fecha_inicio_semana"] <= FECHA_FIN_TRAIN
df_train = df[mask_train].copy()

print(f"Filas de train para calculo de ABC/XYZ: {len(df_train):,}")

# ABC: por valor total acumulado en train
valor_por_sku = df_train.groupby("item")["valor_bruto_total"].sum().sort_values(ascending=False)
valor_total = valor_por_sku.sum()
valor_por_sku_pct = (valor_por_sku.cumsum() / valor_total * 100)

def clasificar_abc(pct_acum):
    if pct_acum <= 80:
        return "A"
    elif pct_acum <= 95:
        return "B"
    else:
        return "C"

clase_abc = valor_por_sku_pct.apply(clasificar_abc)
clase_abc.name = "clase_abc"
print("\nDistribucion de clase ABC:")
print(clase_abc.value_counts())

Filas de train para calculo de ABC/XYZ: 386,661

Distribucion de clase ABC:
clase_abc
C    1688
B     318
A      84
Name: count, dtype: int64


In [15]:
# XYZ: por coeficiente de variacion de cantidad semanal (solo train)
cv_por_sku = df_train.groupby("item")["cantidad_total"].agg(["mean", "std"])
cv_por_sku["cv"] = cv_por_sku["std"] / cv_por_sku["mean"].replace(0, np.nan)
cv_por_sku["cv"] = cv_por_sku["cv"].fillna(99)  # SKUs con media=0 se clasifican como Z

def clasificar_xyz(cv):
    if cv <= 0.5:
        return "X"
    elif cv <= 1.0:
        return "Y"
    else:
        return "Z"

cv_por_sku["clase_xyz"] = cv_por_sku["cv"].apply(clasificar_xyz)
print("Distribucion de clase XYZ:")
print(cv_por_sku["clase_xyz"].value_counts())

Distribucion de clase XYZ:
clase_xyz
Z    2078
Y      11
X       1
Name: count, dtype: int64


In [16]:
# Merge de las clases al dataset completo
df = df.merge(clase_abc.to_frame(), left_on="item", right_index=True, how="left")
df = df.merge(cv_por_sku[["clase_xyz"]], left_on="item", right_index=True, how="left")

# SKUs sin clasificacion (no estaban en train): se marcan como C/Z
df["clase_abc"] = df["clase_abc"].fillna("C")
df["clase_xyz"] = df["clase_xyz"].fillna("Z")

# Codificacion numerica para el modelo
df["clase_abc_num"] = df["clase_abc"].map({"A": 1, "B": 2, "C": 3})
df["clase_xyz_num"] = df["clase_xyz"].map({"X": 1, "Y": 2, "Z": 3})

print("Clases ABC/XYZ asignadas al dataset completo.")
df[["item", "clase_abc", "clase_xyz", "clase_abc_num", "clase_xyz_num"]].drop_duplicates(subset="item").head()

Clases ABC/XYZ asignadas al dataset completo.


,item,clase_abc,clase_xyz,clase_abc_num,clase_xyz_num
0,000001,B,Z,2,3
354,000002,C,Z,3,3
708,000003,B,Z,2,3
1062,000004,C,Z,3,3
1298,000005,A,Z,1,3


## 8. Estadísticas por SKU (solo con train)

Media, mediana, std, min y max de cada SKU en el set de entrenamiento. Estas estadísticas dan al modelo contexto sobre el comportamiento histórico de cada producto.

In [17]:
# Estadisticas por SKU calculadas solo sobre el train
stats_sku = (
    df_train.groupby("item")["cantidad_total"]
    .agg([
        ("sku_mean", "mean"),
        ("sku_median", "median"),
        ("sku_std", "std"),
        ("sku_min", "min"),
        ("sku_max", "max"),
    ])
    .reset_index()
)

# Rellenar nulos en std (SKUs con una sola observacion)
stats_sku["sku_std"] = stats_sku["sku_std"].fillna(0)

# Merge al dataset completo
df = df.merge(stats_sku, on="item", how="left")

# SKUs nuevos (sin observaciones en train) reciben 0
for col in ["sku_mean", "sku_median", "sku_std", "sku_min", "sku_max"]:
    df[col] = df[col].fillna(0)

print("Estadisticas por SKU creadas: sku_mean, sku_median, sku_std, sku_min, sku_max")
stats_sku.head()

Estadisticas por SKU creadas: sku_mean, sku_median, sku_std, sku_min, sku_max


,item,sku_mean,sku_median,sku_std,sku_min,sku_max
0,000001,2.5652,1.0000,4.9477,0.0000,41.0000
1,000002,7.0254,4.0000,9.9162,0.0000,77.0000
2,000003,11.6739,9.0000,14.5506,0.0000,95.0000
3,000004,38.9321,17.0000,42.9822,0.0000,169.0000
4,000005,15.7138,9.0000,19.5255,0.0000,103.0000


## 9. Codificación de variables categóricas

Para LightGBM/XGBoost transformamos las categóricas a códigos numéricos.

In [18]:
# Agregar metadatos del producto
df = df.merge(
    dim_producto[["item", "nombre_linea_n1", "nombre_linea_n2", "proveedor_codigo"]],
    on="item",
    how="left",
)

# Codificacion de categoricas (Label Encoding simple)
for col in ["nombre_linea_n1", "nombre_linea_n2", "proveedor_codigo", "centro_operacion"]:
    df[col] = df[col].fillna("DESCONOCIDO").astype(str)
    df[f"{col}_cod"] = df[col].astype("category").cat.codes

print("Categoricas codificadas:")
print("  nombre_linea_n1_cod, nombre_linea_n2_cod, proveedor_codigo_cod, centro_operacion_cod")
df[["nombre_linea_n1", "nombre_linea_n1_cod",
    "centro_operacion", "centro_operacion_cod"]].head()

Categoricas codificadas:
  nombre_linea_n1_cod, nombre_linea_n2_cod, proveedor_codigo_cod, centro_operacion_cod


,nombre_linea_n1,nombre_linea_n1_cod,centro_operacion,centro_operacion_cod
0,SIKA,13,001,0
1,SIKA,13,001,0
2,SIKA,13,001,0
3,SIKA,13,001,0
4,SIKA,13,001,0


## 10. Partición temporal

Aplicamos los cortes definidos en `src/models.py`:
- Train: hasta 2025-09-30
- Validation: 2025-10-01 a 2025-12-31
- Test 2026: 2026-01-01 a 2026-03-31

In [19]:
# Asignar split a cada fila
def asignar_split(fecha):
    if fecha <= FECHA_FIN_TRAIN:
        return "train"
    elif fecha <= FECHA_FIN_VALID:
        return "valid"
    else:
        return "test_2026"

df["split"] = df["fecha_inicio_semana"].apply(asignar_split)

resumen_splits = (
    df.groupby("split")
    .agg(
        filas=("item", "count"),
        skus_distintos=("item", "nunique"),
        semana_min=("fecha_inicio_semana", "min"),
        semana_max=("fecha_inicio_semana", "max"),
    )
    .reindex(["train", "valid", "test_2026"])
)
resumen_splits

,filas,skus_distintos,semana_min,semana_max
split,,,,
train,386661,2090,2024-01-01,2025-09-29
valid,64613,2166,2025-10-06,2025-12-29
test_2026,66572,2310,2026-01-05,2026-03-30


## 11. Validación de calidad final

Antes de guardar, verificamos:
- No hay nulos en las features críticas
- Los lags y rollings tienen nulos esperados solo en las primeras semanas (esto es normal)
- Las clases ABC/XYZ están asignadas
- La cantidad total se preserva

In [20]:
# Columnas de feature finales
features_lags = ["lag_1", "lag_2", "lag_4", "lag_8", "lag_13", "lag_52",
                  "rolling_mean_4", "rolling_mean_13", "rolling_std_4", "diff_1"]
features_calendario = ["mes", "trimestre", "dia_anio", "sin_mes", "cos_mes",
                        "sin_semana", "cos_semana", "num_festivos"]
features_categoricas = ["nombre_linea_n1_cod", "nombre_linea_n2_cod",
                         "proveedor_codigo_cod", "centro_operacion_cod"]
features_clasificacion = ["clase_abc_num", "clase_xyz_num"]
features_estadisticas = ["sku_mean", "sku_median", "sku_std", "sku_min", "sku_max"]

todas_las_features = (features_lags + features_calendario + features_categoricas
                      + features_clasificacion + features_estadisticas)

print(f"Total features creadas: {len(todas_las_features)}")
print(f"  Lags y rollings:   {len(features_lags)}")
print(f"  Calendario:        {len(features_calendario)}")
print(f"  Categoricas:       {len(features_categoricas)}")
print(f"  Clasificacion:     {len(features_clasificacion)}")
print(f"  Estadisticas:      {len(features_estadisticas)}")

Total features creadas: 29
  Lags y rollings:   10
  Calendario:        8
  Categoricas:       4
  Clasificacion:     2
  Estadisticas:      5


In [21]:
# Reporte de nulos (los lags tendran nulos en las primeras semanas de cada SKU - es normal)
nulos = df[todas_las_features].isna().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
calidad = pd.DataFrame({"nulos": nulos, "pct": nulos_pct})
calidad[calidad["nulos"] > 0].sort_values("nulos", ascending=False)

,nulos,pct
lag_52,253415,48.9400
lag_13,66572,12.8600
lag_8,41269,7.9700
lag_4,20747,4.0100
lag_2,10396,2.0100
rolling_std_4,10396,2.0100
lag_1,5201,1.0000
rolling_mean_4,5201,1.0000
rolling_mean_13,5201,1.0000
diff_1,5201,1.0000


In [22]:
# Estrategia: rellenar nulos en lags/rollings con 0
# (interpretacion: si no hay historia previa, asumimos cero ventas previas)
for col in features_lags:
    df[col] = df[col].fillna(0)

# Verificar que ya no hay nulos en features
nulos_finales = df[todas_las_features].isna().sum().sum()
print(f"Nulos finales en features: {nulos_finales}")
assert nulos_finales == 0, "Quedan nulos en las features!"
print("OK: ninguna feature tiene nulos.")

Nulos finales en features: 0
OK: ninguna feature tiene nulos.


## 12. Guardado del dataset

Guardamos en dos formatos:
- **Parquet** en `data/processed/` — para los siguientes notebooks (carga rápida).
- **Tabla MySQL `dataset_features`** — para inspeccionarlo desde Adminer.

In [23]:
# Columnas finales del dataset
columnas_finales = (
    ["item", "centro_operacion", "fecha_inicio_semana", "split",
     "cantidad_total", "valor_bruto_total", "num_transacciones",
     "anio", "semana",
     "clase_abc", "clase_xyz"]
    + todas_las_features
)

dataset = df[columnas_finales].copy()

print(f"Dataset final:")
print(f"  Filas:    {len(dataset):>10,}")
print(f"  Columnas: {len(dataset.columns):>10}")
print(f"  Memoria:  {dataset.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")

Dataset final:
  Filas:       517,846
  Columnas:         40
  Memoria:  267.43 MB


In [24]:
# Guardar en parquet (formato eficiente)
ruta_parquet = DATA_PROCESSED / "dataset_modelable.parquet"
dataset.to_parquet(ruta_parquet, compression="snappy", index=False)
print(f"Guardado parquet: {ruta_parquet.relative_to(ROOT)}")
print(f"Tamaño: {ruta_parquet.stat().st_size / 1024**2:.2f} MB")

Guardado parquet: data/processed/dataset_modelable.parquet
Tamaño: 5.88 MB


In [25]:
# Guardar tambien en MySQL (para acceder desde Adminer/Tableau)
# Cuidado: tabla grande, usamos chunks
print("Guardando en MySQL (tabla dataset_features)...")

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS dataset_features"))

dataset.to_sql(
    "dataset_features",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10_000,
    method="multi",
)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM dataset_features")).scalar()
print(f"Tabla dataset_features cargada: {n:,} filas")

Guardando en MySQL (tabla dataset_features)...
Tabla dataset_features cargada: 517,846 filas


## 13. Resumen y conclusiones

### 13.1 Decisiones aplicadas

- **SKUs activos:** se modelan únicamente los SKUs con venta en los últimos 90 días del periodo, descartando productos descontinuados.
- **Expansión con ceros:** cada combinación SKU × centro tiene todas las semanas desde su primera venta, con 0 en las semanas sin venta.
- **Sin data leakage:** la clasificación ABC/XYZ y las estadísticas por SKU se calculan exclusivamente sobre el set de train.
- **Lags rellenados con 0:** los valores nulos en lags (primeras semanas de cada SKU) se rellenan con cero, asumiendo demanda histórica nula.
- **Partición temporal estricta:** train (91 semanas), validation (13 semanas), test 2026 (13 semanas).

### 13.2 Estadísticas finales

Ejecuta la siguiente celda para ver el resumen numérico final del dataset.

In [26]:
# Resumen ejecutivo final
print("=" * 60)
print("DATASET MODELABLE — RESUMEN FINAL")
print("=" * 60)
print(f"\nFilas:                {len(dataset):>12,}")
print(f"SKUs activos:         {dataset['item'].nunique():>12,}")
print(f"Centros:              {dataset['centro_operacion'].nunique():>12,}")
print(f"Semanas:              {dataset['fecha_inicio_semana'].nunique():>12,}")
print(f"Features predictivas: {len(todas_las_features):>12,}")
print()
print("Particion temporal:")
for split in ["train", "valid", "test_2026"]:
    sub = dataset[dataset["split"] == split]
    print(f"  {split:>10}: {len(sub):>10,} filas ({sub['fecha_inicio_semana'].min().date()} -> {sub['fecha_inicio_semana'].max().date()})")
print()
print("Distribucion clase ABC:")
print(dataset[["item", "clase_abc"]].drop_duplicates()["clase_abc"].value_counts().sort_index())
print()
print("Distribucion clase XYZ:")
print(dataset[["item", "clase_xyz"]].drop_duplicates()["clase_xyz"].value_counts().sort_index())
print()
print("=" * 60)
print("Dataset listo para modelado en notebooks 05+")
print("=" * 60)

DATASET MODELABLE — RESUMEN FINAL

Filas:                     517,846
SKUs activos:                2,310
Centros:                         3
Semanas:                       118
Features predictivas:           29

Particion temporal:
       train:    386,661 filas (2024-01-01 -> 2025-09-29)
       valid:     64,613 filas (2025-10-06 -> 2025-12-29)
   test_2026:     66,572 filas (2026-01-05 -> 2026-03-30)

Distribucion clase ABC:
clase_abc
A      84
B     318
C    1908
Name: count, dtype: int64

Distribucion clase XYZ:
clase_xyz
X       1
Y      11
Z    2298
Name: count, dtype: int64

Dataset listo para modelado en notebooks 05+


### 13.3 Próximos pasos

- **Notebook 04 — ABC/XYZ:** formalización del análisis descriptivo y dashboard (Tema 1).
- **Notebook 05 — Baseline:** modelo de referencia (media móvil 4 semanas).
- **Notebook 06 — LightGBM:** modelo principal de forecasting.
- **Notebook 07 — XGBoost:** modelo de comparación.
- **Notebook 08 — Prophet:** modelo para top 50 SKUs clase A.
- **Notebook 09 — Evaluación final:** comparación de modelos en validation y test 2026.